# 01 - Prompt Design Patterns (提示设计模式)

## 学习目标
- 掌握五种核心提示模式：Zero-shot, Few-shot, Chain-of-Thought (CoT), ReAct, Self-Ask
- 理解每种模式的适用场景和最佳实践
- 能够通过实验对比不同模式的性能差异

## 模式演进路径
```
Zero-shot → Few-shot → CoT → ReAct → Self-Ask
  (简单)     (模式)    (推理)   (工具)   (分解)
```

In [ ]:
# 初始化环境：导入依赖、配置 API、定义工具函数
import os
import json
import time
from typing import Optional
from dataclasses import dataclass, field
from openai import OpenAI

# 配置 OpenAI 客户端
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    base_url=os.environ.get("OPENAI_BASE_URL", None),  # 支持自定义 base_url
)

# 测试连接
print("OpenAI 客户端已初始化")
print(f"模型: {os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}")

In [ ]:
@dataclass
class PromptResult:
    """提示模式执行结果"""
    pattern_name: str       # 模式名称
    system_prompt: str      # 系统提示
    user_prompt: str        # 用户提示
    output: str             # LLM 输出
    latency_ms: float       # 响应延迟（毫秒）
    token_count: int = 0    # 使用的 token 数

def call_llm(
    system_prompt: str,
    user_prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    max_tokens: int = 1024,
) -> tuple[str, float]:
    """
    调用 LLM 并返回输出和延迟。
    
    What: 封装 OpenAI Chat Completion API 调用。
    Why: 统一接口，避免每个模式重复连接代码。
    When: 所有需要调用 LLM 的地方。
    """
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    latency = (time.perf_counter() - start) * 1000  # 转换为毫秒
    output = response.choices[0].message.content or ""
    return output, latency

print("call_llm 函数已定义")

---
## 模式 1: Zero-shot Prompting (零样本提示)

### 原理
直接向模型描述任务，不提供任何示例。依赖模型的预训练知识完成推理。

### 最佳场景
- 简单分类任务（情感分析、意图识别）
- 信息抽取（命名实体、关键词）
- 摘要和改写

### 设计要素
1. **角色设定**：明确模型的角色和身份
2. **任务描述**：清晰界定输入输出格式
3. **约束条件**：列出边界条件和限制
4. **输出格式**：指定期望的输出结构

In [ ]:
# ============================================================
# 模式 1: Zero-shot 演示
# ============================================================

# 情景 A: 情感分类
zero_shot_system_a = """你是一个专业的文本情感分析助手。
你的任务是对用户提供的文本进行情感分类。
输出格式要求：仅返回 JSON 对象，包含以下字段：
  - sentiment: "positive" | "negative" | "neutral"
  - confidence: 0.0 到 1.0 之间的浮点数
  - keywords: 影响情感判断的关键词列表"""

zero_shot_user_a = "这个新出的手机真是太棒了，拍照效果比我之前用的好太多了！"

output_a, latency_a = call_llm(zero_shot_system_a, zero_shot_user_a)
print(f"=== Zero-shot 情感分类 ===")
print(f"输入: {zero_shot_user_a}")
print(f"输出: {output_a}")
print(f"延迟: {latency_a:.1f}ms")

# 情景 B: 信息抽取
zero_shot_system_b = """你是一个信息抽取系统。
从给定的产品描述中抽取以下信息，以 JSON 格式返回：
  - product_name: 产品名称
  - brand: 品牌
  - price: 价格（数字）
  - features: 功能特性列表
  - target_audience: 目标用户群体"""

zero_shot_user_b = """Apple MacBook Pro 14英寸，搭载 M4 Pro 芯片，32GB 统一内存，
1TB SSD，Liquid Retina XDR 显示屏，售价 19999 元。
适合专业创意工作者和软件开发者，支持 Thunderbolt 5 和 Wi-Fi 7。"""

output_b, latency_b = call_llm(zero_shot_system_b, zero_shot_user_b)
print(f"\n=== Zero-shot 信息抽取 ===")
print(f"输入: {zero_shot_user_b[:80]}...")
print(f"输出: {output_b}")
print(f"延迟: {latency_b:.1f}ms")

### Zero-shot 分析

**优势**：
- 零示例成本，开发速度快
- 对常见任务效果好（模型已预训练）
- 延迟最低，适合实时系统

**局限**：
- 对专业领域任务效果差
- 输出格式不稳定
- 无法处理复杂推理

**适用判断**：
```
任务是否属于常见 NLP 任务？
  是 → 尝试 Zero-shot → 输出是否稳定？
    是 → 使用 Zero-shot
    否 → 升级到 Few-shot
  否 → 直接使用 Few-shot
```

---
## 模式 2: Few-shot Prompting (少样本提示)

### 原理
在提示中提供 2-5 个（输入，输出）示例，让模型通过模式匹配学习任务格式和期望。

### 关键发现
- **格式比数量更重要**：一致的格式比更多示例更有效
- **示例质量 > 示例数量**：2-3 个高质量示例优于 10 个平庸示例
- **多样性覆盖**：示例应覆盖不同的输出类别和边界情况
- **标签平衡**：多分类任务中每个类别至少 1 个示例

In [ ]:
# ============================================================
# 模式 2: Few-shot 演示
# ============================================================

# Few-shot 模板：任务说明 + 格式清晰的示例
few_shot_system = """你是一个任务分类助手。
将用户的请求归类为以下类别之一：
  - booking: 预订服务（酒店、餐厅、机票）
  - inquiry: 信息查询（价格、时间、政策）
  - complaint: 投诉和问题反馈
  - modification: 修改已有订单
  - cancellation: 取消服务

请严格以 JSON 格式返回：
  {"category": "<类别>", "reason": "<判断依据>"}

以下是几个例子："""

few_shot_user = """
示例 1:
输入: 我想订一个下周五的北京到上海的高铁票
输出: {"category": "booking", "reason": "用户明确表示要预订高铁票，属于预订服务"}

示例 2:
输入: 你们酒店的退房时间是几点
输出: {"category": "inquiry", "reason": "用户在询问政策信息，属于信息查询"}

示例 3:
输入: 我上周订的房间空调坏了，体验非常差
输出: {"category": "complaint", "reason": "用户在反馈服务质量问题，属于投诉"}

示例 4:
输入: 我之前订的机票能改签到后天吗
输出: {"category": "modification", "reason": "用户想修改已有订单，属于订单修改"}

现在请对以下输入进行分类：
输入: 我订的明天晚上的包厢能取消吗，临时有事去不了了
输出:"""

output_fs, latency_fs = call_llm(few_shot_system, few_shot_user, max_tokens=256)
print(f"=== Few-shot 任务分类 ===")
print(f"输出: {output_fs}")
print(f"延迟: {latency_fs:.1f}ms")

In [ ]:
# ============================================================
# Zero-shot vs Few-shot 对比实验
# ============================================================

# 测试数据：故意包含边界情况
test_cases = [
    {"id": 1, "input": "帮我查一下明天飞三亚最便宜的航班", "expected": "inquiry"},
    {"id": 2, "input": "我上周订的房我想改一下入住日期", "expected": "modification"},
    {"id": 3, "input": "你们的服务太差了我要投诉", "expected": "complaint"},
    {"id": 4, "input": "不想要了帮我退了吧", "expected": "cancellation"},  # 边界：隐含取消意图
    {"id": 5, "input": "能帮我看下这个酒店还有没有空房，有的话帮我订一间", "expected": "booking"},  # 复合意图
]

def test_zero_shot():
    """对测试集运行 Zero-shot 分类并统计准确率"""
    correct = 0
    for tc in test_cases:
        system_prompt = """将用户请求分类为 booking, inquiry, complaint, modification, cancellation。
只返回 JSON: {"category": "...", "reason": "..."}"""
        output, _ = call_llm(system_prompt, tc["input"], max_tokens=128)
        try:
            result = json.loads(output.strip().lstrip("```json").rstrip("```").strip())
            cat = result.get("category", "").lower()
        except json.JSONDecodeError:
            cat = ""
        correct += 1 if cat == tc["expected"] else 0
        status = "✓" if cat == tc["expected"] else "✗"
        print(f"  {status} 案例{tc['id']}: 预期={tc['expected']}, 实际={cat}")
    return correct / len(test_cases)

def test_few_shot():
    """对测试集运行 Few-shot 分类并统计准确率"""
    correct = 0
    for tc in test_cases:
        system_prompt = few_shot_system
        user_prompt = f"""
示例 1:
输入: 我想订一个下周五的北京到上海的高铁票
输出: {{"category": "booking", "reason": "用户明确表示要预订高铁票"}}

示例 2:
输入: 你们酒店的退房时间是几点
输出: {{"category": "inquiry", "reason": "用户询问政策信息"}}

示例 3:
输入: 我上周订的房间空调坏了，体验非常差
输出: {{"category": "complaint", "reason": "用户反馈服务质量问题"}}

示例 4:
输入: 我之前订的机票能改签到后天吗
输出: {{"category": "modification", "reason": "用户想修改已有订单"}}

现在请对以下输入进行分类：
输入: {tc['input']}
输出:"""
        output, _ = call_llm(system_prompt, user_prompt, max_tokens=128)
        try:
            result = json.loads(output.strip().lstrip("```json").rstrip("```").strip())
            cat = result.get("category", "").lower()
        except json.JSONDecodeError:
            cat = ""
        correct += 1 if cat == tc["expected"] else 0
        status = "✓" if cat == tc["expected"] else "✗"
        print(f"  {status} 案例{tc['id']}: 预期={tc['expected']}, 实际={cat}")
    return correct / len(test_cases)

print("=== Zero-shot 结果 ===")
zs_acc = test_zero_shot()
print(f"准确率: {zs_acc:.0%}")

print("\n=== Few-shot 结果 ===")
fs_acc = test_few_shot()
print(f"准确率: {fs_acc:.0%}")

print(f"\n=== 对比 ===")
print(f"Zero-shot:  {zs_acc:.0%}")
print(f"Few-shot:   {fs_acc:.0%}")
print(f"提升:       {fs_acc - zs_acc:+.0%}")

---
## 模式 3: Chain-of-Thought (CoT, 思维链推理)

### 原理
引导模型在给出最终答案前，先"展示推理过程"。通过 `"Let's think step by step"` 激活模型的分步推理能力。

### 何时使用
- 数学计算和多步推理
- 逻辑推理和常识推理
- 需要中间步骤才能得出正确答案的任务

### 何时不用
- 简单分类任务（CoT 反而增加错误）
- 已有明确公式的计算（直接计算更快）
- 延迟敏感的场景

In [ ]:
# ============================================================
# 模式 3: Chain-of-Thought 演示
# ============================================================

# 标准提示（无 CoT）
standard_system = "你是一个数学问题求解助手。请直接给出答案。"
standard_user = """
问题：一个商店以 120 元的价格买进了一批商品，以 150 元的价格卖出。
后来发现有 20% 的商品有瑕疵，这些商品以进价的 70% 清仓处理。
商店的最终利润率是多少？
"""

print("=== 标准提示（无 CoT）===")
output_std, lat_std = call_llm(standard_system, standard_user)
print(f"输出: {output_std}")
print(f"延迟: {lat_std:.1f}ms")

# CoT 提示
cot_system = """你是一个数学问题求解助手。
对于每个问题，你必须按以下步骤推理：
1. 列出已知条件和未知量
2. 一步一步计算中间结果
3. 验证计算的正确性
4. 给出最终答案

格式：
【已知条件】...
【推理步骤】...
【验证】...
【最终答案】..."""

cot_user = """
请一步步推理以下问题：

问题：一个商店以 120 元的价格买进了一批商品，以 150 元的价格卖出。
后来发现有 20% 的商品有瑕疵，这些商品以进价的 70% 清仓处理。
商店的最终利润率是多少？
"""

print("\n=== CoT 提示 ===")
output_cot, lat_cot = call_llm(cot_system, cot_user, max_tokens=1024)
print(f"输出: {output_cot}")
print(f"延迟: {lat_cot:.1f}ms")

In [ ]:
# ============================================================
# CoT 准确率对比实验
# ============================================================

# 多步推理测试题
math_problems = [
    {
        "id": 1,
        "question": "小明有 350 元，他想买 5 本书，每本原价 80 元。书店打 8.5 折，他够钱吗？找零多少？",
        "expected_answer_pattern": ["够", "10"]  # 实际 340 元，够，找零 10
    },
    {
        "id": 2,
        "question": "一个水池，A 管单独注满需要 4 小时，B 管单独注满需要 6 小时，C 管单独放空需要 12 小时。三管同时打开，几小时注满？",
        "expected_answer_pattern": ["3"]  # 1/4 + 1/6 - 1/12 = 1/3, 3 小时
    },
    {
        "id": 3,
        "question": "甲和乙同时从两地相向而行，甲每小时走 5 公里，乙每小时走 4 公里。3 小时后两人相遇。两地相距多少公里？",
        "expected_answer_pattern": ["27"]  # (5+4) * 3 = 27
    },
]

def check_answer(output: str, patterns: list[str]) -> bool:
    """检查输出中是否包含所有期望的答案模式"""
    return all(p in output for p in patterns)

print("=== CoT 准确率对比 ===")
correct_std = 0
correct_cot = 0

for prob in math_problems:
    # 标准提示
    out_std, _ = call_llm(standard_system, f"问题：{prob['question']}", max_tokens=256)
    ok_std = check_answer(out_std, prob["expected_answer_pattern"])
    if ok_std:
        correct_std += 1
    print(f"  标准提示 问题{prob['id']}: {'✓' if ok_std else '✗'} | 输出: {out_std[:80]}...")
    
    # CoT 提示
    cot_prompt = f"""请一步步推理以下问题：

问题：{prob['question']}

【已知条件】
【推理步骤】
【最终答案】"""
    out_cot, _ = call_llm(cot_system, cot_prompt, max_tokens=1024)
    ok_cot = check_answer(out_cot, prob["expected_answer_pattern"])
    if ok_cot:
        correct_cot += 1
    print(f"  CoT 提示   问题{prob['id']}: {'✓' if ok_cot else '✗'} | 输出: {out_cot[:80]}...")
    print()

print(f"标准提示准确率: {correct_std}/{len(math_problems)} = {correct_std/len(math_problems):.0%}")
print(f"CoT 提示准确率:  {correct_cot}/{len(math_problems)} = {correct_cot/len(math_problems):.0%}")
print(f"提升: {correct_cot/len(math_problems) - correct_std/len(math_problems):+.0%}")

---
## 模式 4: ReAct (Reasoning + Acting, 推理+行动)

### 原理
ReAct 将推理和行动交织在 Thought → Action → Observation 循环中：
- **Thought（思考）**：分析当前状态，决定下一步
- **Action（行动）**：执行具体操作（调用工具、查询、计算）
- **Observation（观察）**：获取行动结果，更新状态

### 关键
ReAct 不是"先想完再做"，而是"想一步、做一步、看一步"，形成认知闭环。

In [ ]:
# ============================================================
# 模式 4: ReAct 演示（模拟工具调用）
# ============================================================

# 模拟的工具函数
def search_knowledge_base(query: str) -> str:
    """模拟知识库搜索"""
    kb = {
        "python": "Python 3.12 发布于 2023 年 10 月 2 日。",
        "地球": "地球到太阳的平均距离约为 1.496 亿公里（即 1 天文单位 AU）。",
        "光合作用": "光合作用公式: 6CO₂ + 6H₂O + 光能 → C₆H₁₂O₆ + 6O₂",
        "大语言模型": "大语言模型基于 Transformer 架构，通过自注意力机制处理文本。",
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return f"未找到关于 '{query}' 的信息。"

def calculate(expression: str) -> str:
    """模拟计算器"""
    try:
        result = eval(expression)  # 仅用于演示，生产环境勿用 eval
        return str(result)
    except Exception as e:
        return f"计算错误: {e}"

# ReAct 系统提示
react_system = """你是一个使用 ReAct 框架的智能助手。
你可以使用以下工具：
  - search(query): 搜索知识库
  - calculate(expression): 执行数学计算

请严格按以下格式回答，每个步骤一行：
  Thought: <你的推理>
  Action: search(<查询>) 或 calculate(<表达式>)
  Observation: <工具返回结果>
  ... (可重复)
  Thought: <最终推理>
  Final Answer: <最终答案>

如果不需要工具，直接给出 Final Answer。"""

react_user = """
问题：地球到太阳的距离是多少万公里？另外，如果光速是 30万公里/秒，光从太阳到地球需要多少秒？
"""

print("=== ReAct 推理过程（第一轮）===")
# 第一轮：模型决定使用哪个工具
output_r1, _ = call_llm(react_system, react_user, max_tokens=512)
print(output_r1)

# 模拟工具执行
print("\n--- 模拟工具执行 ---")
# 执行 search
search_result = search_knowledge_base("地球 太阳 距离")
print(f"search('地球到太阳的距离') -> {search_result}")

# 执行 calculate
calc_result = calculate("1.496 / 30")
print(f"calculate('1.496/30') -> {calc_result} 秒")

In [ ]:
# ============================================================
# ReAct 完整循环（带真实工具执行）
# ============================================================

def react_loop(question: str, max_steps: int = 5) -> str:
    """
    执行完整的 ReAct 循环。
    
    What: 实现 Thought→Action→Observation 循环直到获得最终答案。
    Why: 展示真正的 ReAct 工作流程，而非仅仅让 LLM 模拟。
    When: 构建 Agent 系统和工具调用链时。
    """
    conversation = [{"role": "system", "content": react_system}]
    conversation.append({"role": "user", "content": f"问题：{question}"})
    
    for step in range(max_steps):
        print(f"\n--- 第 {step + 1} 步 ---")
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=conversation,
            temperature=0.0,
            max_tokens=512,
        )
        text = response.choices[0].message.content or ""
        print(f"模型输出:\n{text}")
        
        # 检查是否有 Action
        if "Action:" in text:
            # 解析 Action
            import re
            action_match = re.search(r'Action:\s*(\w+)\(([^)]*)\)', text)
            if action_match:
                tool_name = action_match.group(1)
                tool_arg = action_match.group(2).strip('").strip("'")
                
                # 执行工具
                if tool_name == "search":
                    result = search_knowledge_base(tool_arg)
                elif tool_name == "calculate":
                    result = calculate(tool_arg)
                else:
                    result = f"未知工具: {tool_name}"
                
                print(f"工具执行: {tool_name}({tool_arg}) -> {result}")
                
                # 将结果反馈给模型
                conversation.append({"role": "assistant", "content": text})
                conversation.append({"role": "user", "content": f"Observation: {result}"})
                continue
        
        # 检查是否有 Final Answer
        if "Final Answer:" in text:
            final = text.split("Final Answer:")[1].strip()
            return final
        
        # 没有 Action 也没有 Final Answer → 要求模型继续
        conversation.append({"role": "assistant", "content": text})
        conversation.append({"role": "user", "content": "请继续，如果已有答案请给出 Final Answer。"})
    
    return "达到最大步数。"

# 测试完整的 ReAct 循环
question = "太阳光线到达地球需要多长时间？（使用光速 30万km/s）"
answer = react_loop(question, max_steps=3)
print(f"\n{'='*50}")
print(f"最终答案: {answer}")

---
## 模式 5: Self-Ask (自问自答 / 问题分解)

### 原理
Self-Ask 让模型将复杂问题分解为一系列子问题，逐一解答后组合成最终答案。
与 ReAct 不同，Self-Ask 的"工具"是模型自身的推理能力。

### 适用场景
- 需要多步信息组合的问答
- 对比分析任务（先各自分析，再对比）
- 法律、合规分析（逐条审查）

In [ ]:
# ============================================================
# 模式 5: Self-Ask 演示
# ============================================================

self_ask_system = """你是一个分析助手。对于复杂问题，你必须先将其分解为子问题，
然后逐一回答，最后综合给出结论。

请按以下格式回答：
【问题分解】列出所有子问题
【子问题 1】<问题文本> → <答案>
【子问题 2】<问题文本> → <答案>
【综合分析】将子答案组合成完整回答
【最终结论】简明扼要的最终答案"""

self_ask_user_complex = """
请分析和比较 Python 和 JavaScript 在以下方面的差异：
1. 异步编程模型
2. 类型系统
3. 包管理工具

最后给出两种语言各适合什么场景的建议。
"""

print("=== Self-Ask 复杂问题分解 ===")
output_sa, lat_sa = call_llm(self_ask_system, self_ask_user_complex, max_tokens=2048)
print(output_sa)
print(f"\n延迟: {lat_sa:.1f}ms")

In [ ]:
# ============================================================
# Self-Ask vs 直接回答对比
# ============================================================

comparison_question = """
一家公司有 3 个部门：
- 部门A：50人，平均工资 12000，有 10 个人薪资高于 15000
- 部门B：30人，平均工资 15000，有 5 个人薪资高于 18000
- 部门C：20人，平均工资 20000，有 8 个人薪资高于 22000

请计算：
1. 全公司平均工资
2. 全公司高薪人员（高于 15000）占比
3. 哪个部门的薪酬公平性最好（薪资高于部门均值的比例最低）
"""

print("=== 直接回答 ===")
direct_output, direct_lat = call_llm("直接回答以下问题", comparison_question, max_tokens=1024)
print(direct_output)
print(f"延迟: {direct_lat:.1f}ms")

print("\n=== Self-Ask 分步回答 ===")
sa_output, sa_lat = call_llm(self_ask_system, comparison_question, max_tokens=2048)
print(sa_output)
print(f"延迟: {sa_lat:.1f}ms")

## 五种模式对比总结

| 模式 | 推理深度 | Token 消耗 | 延迟 | 最佳场景 |
|------|----------|-----------|------|---------|
| Zero-shot | 浅 | 低 | 低 | 简单分类、抽取、摘要 |
| Few-shot | 浅-中 | 中 | 中 | 格式化输出、自定义分类 |
| CoT | 中-深 | 中 | 中 | 数学推理、逻辑题 |
| ReAct | 深 | 高 | 高 | 需要工具调用、多步查询 |
| Self-Ask | 深 | 高 | 高 | 复杂分析、对比、审查 |

### 模式选择决策树

```
任务需要外部工具吗？
  ├─ 是 → ReAct
  └─ 否 → 需要多步推理吗？
        ├─ 是 → 问题可以分解为独立子问题？
        │     ├─ 是 → Self-Ask
        │     └─ 否 → CoT
        └─ 否 → 输出格式复杂/特殊？
              ├─ 是 → Few-shot (2-3个格式化示例)
              └─ 否 → Zero-shot
```

---
## 练习：为场景选择最佳模式

对于以下 5 个场景，选择最合适的提示模式并说明理由。

In [ ]:
# ============================================================
# 练习：模式选择
# ============================================================

scenarios = [
    {
        "id": 1,
        "scenario": "构建一个智能客服机器人，需要查询订单状态、修改订单、处理退款等。",
        "hint": "需要调用多个后端 API，每个 API 的结果可能影响后续决策。"
    },
    {
        "id": 2,
        "scenario": "对 10000 条新闻标题进行情感分类（正面/负面/中性），要求快速且成本低。",
        "hint": "分类任务简单明确，批量处理。"
    },
    {
        "id": 3,
        "scenario": "将非结构化的合同文本转换为结构化的 JSON，字段包括：甲方、乙方、金额、期限、违约条款。",
        "hint": "输出格式复杂且字段很多，需要模型理解输出范式。"
    },
    {
        "id": 4,
        "scenario": "分析三份财务报表（资产负债表、利润表、现金流量表），找出公司可能存在的财务风险并撰写风险报告。",
        "hint": "多文档综合分析，需要多维度交叉验证。"
    },
    {
        "id": 5,
        "scenario": "一个数学辅导应用，专门解答初中几何证明题。",
        "hint": "几何证明需要一步步推导，每步有逻辑依据。"
    },
]

# 参考答案（学生自行完成后再查看）
answers = {
    1: ("ReAct", "需要多个 API 调用，每次调用结果影响后续决策，符合 ReAct 的 Thought→Action→Observation 模式"),
    2: ("Zero-shot", "情感分类是模型预训练已掌握的基础任务，10000条批量处理需要低延迟低成本"),
    3: ("Few-shot", "合同格式多种多样，需要2-3个格式化示例让模型理解输出范式，结构化输出配合 JSON Schema"),
    4: ("Self-Ask", "三份报表需要分别分析（子问题）再综合判断，问题可以清晰分解为独立子任务"),
    5: ("Chain-of-Thought", "几何证明天生需要分步推导，每步有定理依据，CoT 能展示完整证明链"),
}

print("请为以下每个场景选择最合适的提示模式：\n")
for s in scenarios:
    print(f"场景 {s['id']}: {s['scenario']}")
    print(f"  提示: {s['hint']}")
    print()

print("="*60)
print("参考答案（请在自行思考后再查看）：\n")
for sid, (mode, reason) in answers.items():
    print(f"场景 {sid}: {mode}")
    print(f"  理由: {reason}\n")

In [ ]:
# ============================================================
# 加分练习：实现模式路由函数
# ============================================================

def route_prompt(
    task: str,
    input_text: str,
    pattern: str = "auto",
    examples: list[dict] | None = None,
) -> str:
    """
    根据任务和模式，自动构建提示并调用 LLM。
    
    What: 统一的提示路由函数，根据 pattern 参数选择不同的提示策略。
    Why: 避免在业务代码中散布提示构建逻辑，集中管理和复用。
    When: 构建需要灵活切换提示策略的 LLM 应用时。
    
    Args:
        task: 任务描述
        input_text: 输入文本
        pattern: 提示模式 ("zero-shot" | "few-shot" | "cot" | "react" | "self-ask" | "auto")
        examples: Few-shot 示例列表，每个示例 {"input": ..., "output": ...}
    
    Returns:
        LLM 响应文本
    """
    if pattern == "auto":
        # 自动选择模式（简单启发式）
        if examples and len(examples) > 0:
            pattern = "few-shot"
        elif any(kw in task for kw in ["计算", "推导", "证明", "求解"]):
            pattern = "cot"
        else:
            pattern = "zero-shot"
    
    # 根据模式构建提示
    if pattern == "zero-shot":
        system = task
        user = input_text
    elif pattern == "few-shot":
        ex_str = "\n".join(
            f"示例 {i+1}:\n输入: {ex['input']}\n输出: {ex['output']}"
            for i, ex in enumerate(examples or [])
        )
        system = task
        user = f"{ex_str}\n\n现在请处理：\n输入: {input_text}\n输出:"
    elif pattern == "cot":
        system = task + "\n请逐步推理，展示完整的推理过程。"
        user = f"请一步步推理：{input_text}"
    elif pattern == "self-ask":
        system = task + "\n请将问题分解为子问题逐一解答，最后综合。"
        user = f"请分析并回答：{input_text}"
    else:
        system = task
        user = input_text
    
    output, _ = call_llm(system, user, max_tokens=1024)
    return output

# 测试路由函数
print("=== 测试提示路由函数 ===\n")

# Auto 模式 - 应该选择 zero-shot（简单分类）
r1 = route_prompt(
    "判断以下评论的情感：正面、负面或中性。只返回一个词。",
    "这个产品还行吧，没有想象的那么好。",
    pattern="auto"
)
print(f"Auto (简单分类): {r1}\n")

# Few-shot 模式
r2 = route_prompt(
    "将用户反馈转换为 JSON 格式，包含 severity 和 summary。",
    "你们的 APP 一直闪退！根本用不了！",
    pattern="few-shot",
    examples=[
        {"input": "页面加载太慢了", "output": '{"severity": "medium", "summary": "性能问题：页面加载速度慢"}'},
        {"input": "账号被盗了赶紧处理", "output": '{"severity": "critical", "summary": "安全问题：账号被盗"}'},
    ]
)
print(f"Few-shot: {r2}\n")

# CoT 模式
r3 = route_prompt(
    "你是一个数学助手。",
    "如果一个数是 7 的倍数，且比 50 大但比 70 小，这个数是多少？",
    pattern="cot"
)
print(f"CoT: {r3}")

print("\n提示路由函数测试完毕！")

## 本节小结

1. **Zero-shot** 是起点，对常见 LLM 任务足够
2. **Few-shot** 通过格式示范提升输出稳定性，质量 > 数量
3. **CoT** 激活推理链，适合多步推理但增加延迟
4. **ReAct** 适合需要外部工具的 Agent 场景
5. **Self-Ask** 适合可分解的复杂分析任务
6. **模式选择** 取决于：是否有工具、推理深度、输出复杂度、延迟要求